<a href="https://colab.research.google.com/github/tanvi1911/Multiple_Regression_Analysis_on_Diabetes_Dataset/blob/main/Multiple_Linear_Regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np

import plotly.express as px
import plotly.graph_objects as go

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [6]:
# Load all CSV files
import glob

files = glob.glob("data/preprocessed/*.csv")

dfs = [pd.read_csv(f, sep=",", engine="python") for f in files]

df = pd.concat(dfs, ignore_index=True)

# Drop accidental extra column if present
df = df.loc[:, ~df.columns.str.contains(";")]

print(df.shape)
print(df.columns)
display(df.head())

(309392, 8)
Index(['time', 'glucose', 'calories', 'heart_rate', 'steps', 'basal_rate',
       'bolus_volume_delivered', 'carb_input'],
      dtype='str')


,time,glucose,calories,heart_rate,steps,basal_rate,bolus_volume_delivered,carb_input
0,2018-06-13T18:40:00,332.0,6.3595,82.322835,34.0,0.091667,0.0,0.0
1,2018-06-13T18:45:00,326.0,7.7280,83.740157,0.0,0.091667,0.0,0.0
2,2018-06-13T18:50:00,330.0,4.7495,80.525180,0.0,0.091667,0.0,0.0
3,2018-06-13T18:55:00,324.0,6.3595,89.129032,20.0,0.091667,0.0,0.0
4,2018-06-13T19:00:00,306.0,5.1520,92.495652,0.0,0.075000,0.0,0.0


In [13]:
# Convert time column to datetime
df["time"] = pd.to_datetime(df["time"])

# Ensure numeric columns are numeric
numeric_cols = [
    "glucose",
    "calories",
    "heart_rate",
    "steps",
    "basal_rate",
    "bolus_volume_delivered",
    "carb_input"
]

df[numeric_cols] = df[numeric_cols].apply(pd.to_numeric)

# Check types
df.dtypes

time                      datetime64[us]
glucose                          float64
calories                         float64
heart_rate                       float64
steps                            float64
basal_rate                       float64
bolus_volume_delivered           float64
carb_input                       float64
patient_id                           str
dtype: object

In [14]:
# Check missing values
missing = df.isnull().sum()

print(missing)

time                      285156
glucose                   285156
calories                  285156
heart_rate                285156
steps                     285156
basal_rate                285156
bolus_volume_delivered    285156
carb_input                285156
patient_id                     0
dtype: int64


In [15]:
# Drop rows that are completely empty
df = df.dropna(how="all")

# Check missing values again
missing = df.isnull().sum()

print(df.shape)
print(missing)

(309392, 9)
time                      285156
glucose                   285156
calories                  285156
heart_rate                285156
steps                     285156
basal_rate                285156
bolus_volume_delivered    285156
carb_input                285156
patient_id                     0
dtype: int64


In [16]:
# Sort data by time
df = df.sort_values("time")

# Reset index
df = df.reset_index(drop=True)

df.head()

,time,glucose,calories,heart_rate,steps,basal_rate,bolus_volume_delivered,carb_input,patient_id
0,2018-06-13 18:40:00,332.0,6.3595,82.322835,34.0,0.091667,0.0,0.0,preprocessed\HUPA0001P
1,2018-06-13 18:45:00,326.0,7.7280,83.740157,0.0,0.091667,0.0,0.0,preprocessed\HUPA0001P
2,2018-06-13 18:50:00,330.0,4.7495,80.525180,0.0,0.091667,0.0,0.0,preprocessed\HUPA0001P
3,2018-06-13 18:55:00,324.0,6.3595,89.129032,20.0,0.091667,0.0,0.0,preprocessed\HUPA0001P
4,2018-06-13 19:00:00,306.0,5.1520,92.495652,0.0,0.075000,0.0,0.0,preprocessed\HUPA0001P


In [19]:
import os

dfs = []

for f in files:
    patient_id = os.path.basename(f).replace(".csv", "")
    
    temp = pd.read_csv(f, sep=",", engine="python")
    temp["patient_id"] = patient_id
    
    dfs.append(temp)

df = pd.concat(dfs, ignore_index=True)

# Drop accidental extra column if present
df = df.loc[:, ~df.columns.str.contains(";")]

In [20]:
df.head()

,time,glucose,calories,heart_rate,steps,basal_rate,bolus_volume_delivered,carb_input,patient_id
0,2018-06-13T18:40:00,332.0,6.3595,82.322835,34.0,0.091667,0.0,0.0,HUPA0001P
1,2018-06-13T18:45:00,326.0,7.7280,83.740157,0.0,0.091667,0.0,0.0,HUPA0001P
2,2018-06-13T18:50:00,330.0,4.7495,80.525180,0.0,0.091667,0.0,0.0,HUPA0001P
3,2018-06-13T18:55:00,324.0,6.3595,89.129032,20.0,0.091667,0.0,0.0,HUPA0001P
4,2018-06-13T19:00:00,306.0,5.1520,92.495652,0.0,0.075000,0.0,0.0,HUPA0001P


In [23]:
print(df["patient_id"].nunique())

25


In [24]:
print(len(files))

25


In [25]:
# Create future glucose target (30 minutes ahead)

df["glucose_future"] = (
    df.groupby("patient_id")["glucose"]
    .shift(-6)
)

df[["time", "glucose", "glucose_future"]].head(10)

,time,glucose,glucose_future
0,2018-06-13T18:40:00,332.0,312.0
1,2018-06-13T18:45:00,326.0,293.0
2,2018-06-13T18:50:00,330.0,303.0
3,2018-06-13T18:55:00,324.0,293.0
4,2018-06-13T19:00:00,306.0,271.0
5,2018-06-13T19:05:00,313.0,270.0
6,2018-06-13T19:10:00,312.0,255.0
7,2018-06-13T19:15:00,293.0,221.0
8,2018-06-13T19:20:00,303.0,242.0
9,2018-06-13T19:25:00,293.0,232.0


In [26]:
# Remove rows without future glucose
df = df.dropna(subset=["glucose_future"])

print(df.shape)

(24194, 10)
